In [ ]:
# One MOSS cell, timed. transformers is pinned because MOSS's remote code calls
# generation internals that 5.1 removed.
!pip -q install "transformers>=5.0,<5.1" accelerate speechbrain faster-whisper jiwer pyloudnorm noisereduce num2words silero-vad onnxruntime


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import torch
import transformers

INPUT = Path("/kaggle/input")
found = sorted(INPUT.rglob("repo/scripts/benchmark.py"))
if not found:
    for path in sorted(p for p in INPUT.rglob("*") if p.is_dir())[:40]:
        print("   ", path)
    raise SystemExit("no repo under /kaggle/input - see the directories above")
SOURCE = found[0].parent.parent

REPO = Path("/kaggle/working/repo")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(SOURCE, REPO)

version = tuple(int(part) for part in transformers.__version__.split(".")[:2])
print("transformers:", transformers.__version__)
print("torch       :", torch.__version__)
print("gpus        :", torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
# T4 is Turing: no native bfloat16, so the engine drops to float16. MOSS is the
# only engine here that takes that path, which makes it a suspect for the
# slowness this notebook is measuring.
print("bf16 native :", torch.cuda.is_bf16_supported())
assert (5, 0) <= version < (5, 1), f"MOSS needs transformers 5.0.x, found {transformers.__version__}"
assert torch.cuda.device_count() >= 1, "no GPU - set the accelerator"


def run(*arguments, timeout=None):
    """Run the benchmark and return (exit code, wall-clock seconds)."""
    command = [sys.executable, "scripts/benchmark.py", *arguments]
    print(" ".join(command), flush=True)
    start = time.time()
    try:
        outcome = subprocess.run(command, cwd=REPO, text=True, timeout=timeout)
        code = outcome.returncode
    except subprocess.TimeoutExpired:
        print(f"TIMED OUT after {timeout}s - called off", flush=True)
        code = -1
    elapsed = time.time() - start
    print(f"exit {code} in {elapsed / 60:.1f} min", flush=True)
    return code, elapsed


In [ ]:
# One cell: one speaker, one sentence. The wall clock includes downloading
# roughly 6.1 GB of weights and loading them, so it is an upper bound on what a
# steady-state render costs - which is enough to decide whether 27 cells is
# hours or minutes.
code, elapsed = run("--label", "moss-smoke", "--engine", "moss", "--device", "cuda",
                    "--speakers", "cv02", "--sentences", "s1", timeout=900)

print()
print(f"one cell, end to end: {elapsed / 60:.1f} min")
print(f"that pace over 27 cells: {elapsed * 27 / 3600:.1f} h (before subtracting the one-time download)")

path = REPO / "eval/results/moss-smoke.json"
if path.is_file():
    data = json.loads(path.read_text())
    print("summary  :", data["summary"])
    print("benchmark's own duration_minutes:", data.get("duration_minutes"))
    for row in data["rows"]:
        audio = row.get("seconds")
        print("row      :", {k: row.get(k) for k in
              ("speaker", "sentence", "seconds", "held_out_similarity",
               "character_error_rate", "error")})
        if audio:
            # seconds is the length of the rendered audio, not the time spent.
            # Their ratio is the real-time factor, which is the number that says
            # whether generation is running at a sane rate.
            print(f"rendered {audio:.2f} s of audio; "
                  f"{elapsed / audio:.0f}x slower than real time, load included")
else:
    print("no results file - the render did not get far enough to write one")


In [ ]:
# Now the scale check, after the timing is safely printed. This environment is
# transformers 5.0.x, a different dependency world from every number on record,
# so re-score renders whose answer is already established.
code, elapsed = run("--label", "baseline-moss-env", "--score-only", "--replay-from", "baseline",
                    "--device", "cuda", timeout=1800)
path = REPO / "eval/results/baseline-moss-env.json"
if path.is_file():
    value = json.loads(path.read_text())["summary"]["held_out_mean"]
    drift = value - 0.5196
    print(f"control: {value:.4f} vs 0.5196 known ({drift:+.4f})")
    print("on the established scale" if abs(drift) < 0.0005 else "OFF THE ESTABLISHED SCALE")
else:
    print("control did not produce a results file")


In [ ]:
destination = Path("/kaggle/working/results")
destination.mkdir(exist_ok=True)
for path in (REPO / "eval/results").glob("*.json"):
    shutil.copy2(path, destination / path.name)
renders = Path("/kaggle/working/moss_renders")
renders.mkdir(exist_ok=True)
for path in (REPO / "outputs/benchmark/moss-smoke").glob("*.wav"):
    shutil.copy2(path, renders / path.name)
print("saved:", len(list(renders.glob("*.wav"))), "renders")
